# Detección de Anomalías en el MERVAL — LSTM Autoencoder

**Trabajo Integrador — Inteligencia Artificial**

Pipeline completo: descarga de datos → features → entrenamiento → detección de anomalías.

---
**Antes de ejecutar:** en el menú superior ir a `Entorno de ejecución → Cambiar tipo de entorno de ejecución` y seleccionar **GPU T4** para acelerar el entrenamiento.

## 0. Instalación de dependencias

In [ ]:
# yfinance no viene preinstalado en Colab; el resto (torch, pandas, sklearn, matplotlib) sí
!pip install yfinance --quiet
import importlib, yfinance
print('yfinance:', yfinance.__version__)

## 0b. (Opcional) Montar Google Drive

Si querés guardar los modelos y resultados de forma permanente ejecutá esta celda.
Si no, todo se guarda en `/content/` y se pierde al cerrar la sesión.

In [ ]:
USE_DRIVE = False   # <-- cambiá a True si querés persistencia en Drive

if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    BASE = '/content/drive/MyDrive/merval_anomaly'
else:
    BASE = '/content/merval_anomaly'

from pathlib import Path
for d in ['data/raw', 'data/processed', 'models', 'results']:
    Path(f'{BASE}/{d}').mkdir(parents=True, exist_ok=True)

print('Directorio base:', BASE)

## 1. Imports globales

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
from pathlib import Path

import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
from sklearn.preprocessing import StandardScaler
import yfinance as yf

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', DEVICE)
print('PyTorch:', torch.__version__)

## 2. Descarga de datos

In [ ]:
TICKERS = ['GGAL.BA', 'BMA.BA', 'YPFD.BA', 'PAMP.BA', 'BBAR.BA', 'ALUA.BA', '^MERV']
START   = '2015-01-01'
END     = '2025-12-31'

print(f'Descargando {len(TICKERS)} tickers desde {START} hasta {END}...')
raw = yf.download(TICKERS, start=START, end=END, auto_adjust=True, progress=True)

close  = raw['Close'].copy()
volume = raw['Volume'].copy()

# Rellenar gaps de feriados argentinos (máximo 3 días)
close  = close.ffill(limit=3)
volume = volume.ffill(limit=3)
close  = close.dropna(how='all')
volume = volume.dropna(how='all')

close.to_csv(f'{BASE}/data/raw/close.csv')
volume.to_csv(f'{BASE}/data/raw/volume.csv')

print(f'Guardado: {len(close)} días x {len(close.columns)} tickers')
close.tail(3)

## 3. Ingeniería de features

In [ ]:
WINDOW    = 30
TRAIN_END = '2017-12-31'
VAL_END   = '2018-12-31'
TICKER    = 'GGAL.BA'


def build_features(close, volume, ticker):
    """Construye 8 features para un ticker dado."""
    c     = close[ticker]
    v     = volume[ticker]
    m     = close['^MERV']
    m_ret = m.pct_change()

    df = pd.DataFrame(index=close.index)
    df['ret_1d']      = c.pct_change()
    df['log_ret']     = np.log(c / c.shift(1))
    df['vol_5d']      = df['ret_1d'].rolling(5).std()
    df['vol_20d']     = df['ret_1d'].rolling(20).std()
    df['vol_ratio']   = df['vol_5d'] / (m_ret.rolling(5).std() + 1e-8)
    df['corr_30d']    = df['ret_1d'].rolling(30).corr(m_ret)
    df['zscore']      = (c - c.rolling(90).mean()) / (c.rolling(90).std() + 1e-8)
    df['vol_ratio_v'] = v / (v.rolling(20).mean() + 1)
    return df.dropna()


def make_windows(df, window=WINDOW):
    """Convierte un DataFrame en ventanas deslizantes (N, window, n_features)."""
    X, dates = [], []
    for i in range(len(df) - window):
        X.append(df.values[i : i + window])
        dates.append(df.index[i + window - 1])
    return np.array(X), pd.DatetimeIndex(dates)


def split_and_scale(df):
    """Divide en train/val/test y escala con StandardScaler ajustado solo en train."""
    train_df = df[df.index <= TRAIN_END]
    val_df   = df[(df.index > TRAIN_END) & (df.index <= VAL_END)]
    test_df  = df[df.index > VAL_END]

    scaler = StandardScaler()

    def scale(subset, fit=False):
        if fit:
            scaler.fit(subset.values)
        return pd.DataFrame(scaler.transform(subset.values),
                            index=subset.index, columns=subset.columns)

    train_s = scale(train_df, fit=True)
    val_s   = scale(val_df)
    test_s  = scale(test_df)

    X_train, d_train = make_windows(train_s)
    X_val,   d_val   = make_windows(val_s)
    X_test,  d_test  = make_windows(test_s)

    return X_train, X_val, X_test, d_train, d_val, d_test, scaler


# Procesar GGAL.BA
df = build_features(close, volume, TICKER)
df.to_csv(f'{BASE}/data/processed/features_{TICKER.replace(".", "_")}.csv')

X_train, X_val, X_test, d_train, d_val, d_test, scaler = split_and_scale(df)
print(f'Train: {X_train.shape} | Val: {X_val.shape} | Test: {X_test.shape}')
df.head()

## 4. Arquitectura del modelo

In [ ]:
class LSTMAutoencoder(nn.Module):
    """
    LSTM Autoencoder para detección de anomalías.

    Arquitectura:
      Encoder:    LSTM(64) -> Dropout(0.2)
      Bottleneck: RepeatVector(time_steps)
      Decoder:    LSTM(64, return_seq=True) -> Dropout(0.2) -> Linear(n_features)
    """

    def __init__(self, time_steps=30, n_features=8):
        super().__init__()
        self.time_steps    = time_steps
        self.encoder_lstm  = nn.LSTM(n_features, 64, batch_first=True)
        self.dropout1      = nn.Dropout(0.2)
        self.decoder_lstm  = nn.LSTM(64, 64, batch_first=True)
        self.dropout2      = nn.Dropout(0.2)
        self.output_layer  = nn.Linear(64, n_features)

    def forward(self, x):
        _, (h_n, _) = self.encoder_lstm(x)                          # (1, B, 64)
        encoded     = self.dropout1(h_n[-1])                         # (B, 64)
        repeated    = encoded.unsqueeze(1).expand(-1, self.time_steps, -1)  # (B, T, 64)
        decoded, _  = self.decoder_lstm(repeated)                    # (B, T, 64)
        decoded     = self.dropout2(decoded)
        return self.output_layer(decoded)                            # (B, T, n_features)


# Verificar arquitectura
model_test = LSTMAutoencoder(time_steps=30, n_features=8)
x_test = torch.randn(4, 30, 8)
print('Input:', x_test.shape, '-> Output:', model_test(x_test).shape)
print('Parámetros totales:', sum(p.numel() for p in model_test.parameters()))

## 5. Entrenamiento

In [ ]:
EPOCHS   = 50
BATCH    = 32
PATIENCE = 7


def train_model(X_train, X_val, model_path, epochs=EPOCHS, batch_size=BATCH):
    """Entrena el autoencoder con EarlyStopping sobre val_loss."""
    model     = LSTMAutoencoder(X_train.shape[1], X_train.shape[2]).to(DEVICE)
    optimizer = torch.optim.Adam(model.parameters())

    X_tr = torch.FloatTensor(X_train).to(DEVICE)
    X_v  = torch.FloatTensor(X_val).to(DEVICE)
    loader = DataLoader(TensorDataset(X_tr, X_tr), batch_size=batch_size, shuffle=True)

    best_val  = float('inf')
    patience_ = 0
    history   = {'loss': [], 'val_loss': []}

    for epoch in range(1, epochs + 1):
        model.train()
        losses = []
        for xb, _ in loader:
            optimizer.zero_grad()
            loss = torch.mean(torch.abs(model(xb) - xb))
            loss.backward()
            optimizer.step()
            losses.append(loss.item())

        model.eval()
        with torch.no_grad():
            val_loss = torch.mean(torch.abs(model(X_v) - X_v)).item()

        avg = sum(losses) / len(losses)
        history['loss'].append(avg)
        history['val_loss'].append(val_loss)
        print(f'Epoch {epoch:3d}/{epochs} — loss: {avg:.6f} — val_loss: {val_loss:.6f}')

        if val_loss < best_val:
            best_val  = val_loss
            torch.save(model.state_dict(), model_path)
            patience_ = 0
        else:
            patience_ += 1
            if patience_ >= PATIENCE:
                print(f'EarlyStopping en época {epoch}. Mejor val_loss: {best_val:.6f}')
                model.load_state_dict(torch.load(model_path, weights_only=True))
                break

    return model, history


MODEL_PATH = f'{BASE}/models/best.pt'
model, history = train_model(X_train, X_val, MODEL_PATH)
print(f'\nMejor val_loss: {min(history["val_loss"]):.6f}')

In [ ]:
# Curva de pérdida
plt.figure(figsize=(10, 4))
plt.plot(history['loss'],     label='Train loss')
plt.plot(history['val_loss'], label='Val loss')
plt.xlabel('Época')
plt.ylabel('MAE')
plt.title('Curva de pérdida — LSTM Autoencoder')
plt.legend()
plt.tight_layout()
plt.savefig(f'{BASE}/results/loss_curve.png', dpi=150)
plt.show()

## 6. Evaluación

### Funciones base

In [ ]:
EVENTOS = {
    '2019-08-12': 'PASO 2019\n(-38%)',
    '2020-03-13': 'COVID-19',
    '2023-08-14': 'PASO 2023',
    '2023-12-13': 'Shock Milei',
}


def reconstruction_error(model, X):
    """MAE de reconstrucción por ventana."""
    model.eval()
    with torch.no_grad():
        X_hat = model(torch.FloatTensor(X).to(DEVICE)).cpu().numpy()
    return np.mean(np.abs(X - X_hat), axis=(1, 2))


def compute_threshold(train_errors, sigma=3.0):
    """Umbral = media + sigma * desviacion estándar (calculado sobre train)."""
    return float(train_errors.mean() + sigma * train_errors.std())


train_errors = reconstruction_error(model, X_train)
test_errors  = reconstruction_error(model, X_test)

print(f'Error medio en train: {train_errors.mean():.5f} ± {train_errors.std():.5f}')
print(f'Error medio en test:  {test_errors.mean():.5f} ± {test_errors.std():.5f}')

### Experimento 1 — Variación de sigma

In [ ]:
rows = []
for s in [2.0, 3.0, 4.0]:
    thr = compute_threshold(train_errors, sigma=s)
    n   = int((test_errors > thr).sum())
    rows.append({'sigma': s, 'threshold': round(thr, 5), 'anomalias_detectadas': n})
    print(f'sigma={s} -> umbral={thr:.5f} | anomalias={n}')

sweep_df = pd.DataFrame(rows)
sweep_df.to_csv(f'{BASE}/results/sigma_sweep.csv', index=False)
sweep_df

### Experimento 2 — Gráfico principal (sigma=3)

In [ ]:
sigma     = 3.0
threshold = compute_threshold(train_errors, sigma=sigma)

anomaly_mask  = test_errors > threshold
anomaly_dates = d_test[anomaly_mask]

fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(d_test, test_errors, lw=0.8, color='steelblue',
        alpha=0.85, label='Error de reconstrucción')
ax.fill_between(d_test, test_errors, alpha=0.15, color='steelblue')
ax.axhline(threshold, color='crimson', ls='--', lw=1.5,
           label=f'Umbral {sigma}σ = {threshold:.4f}')
ax.scatter(anomaly_dates, test_errors[anomaly_mask],
           color='crimson', s=18, zorder=5, label='Anomalía detectada')

ylim = ax.get_ylim()
for fecha_str, label in EVENTOS.items():
    fecha = pd.Timestamp(fecha_str)
    if d_test.min() <= fecha <= d_test.max():
        ax.axvline(fecha, color='orange', lw=1.2, alpha=0.9, ls='-.')
        ax.text(fecha, ylim[1] * 0.92, label,
                fontsize=7.5, ha='left', va='top',
                rotation=90, color='darkorange')

ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
ax.xaxis.set_major_locator(mdates.YearLocator())
ax.set_xlabel('Fecha')
ax.set_ylabel('MAE de reconstrucción')
ax.set_title(f'Anomalías detectadas — {TICKER} | Umbral {sigma}σ')
ax.legend(loc='upper left', fontsize=9)
plt.tight_layout()
plt.savefig(f'{BASE}/results/anomalias.png', dpi=150)
plt.show()

print(f'Anomalías detectadas: {len(anomaly_dates)}')
print('Primeras 10:', anomaly_dates.strftime('%Y-%m-%d').tolist()[:10])

### Experimento 3 — Ablation study (2 features vs 8 features)

In [ ]:
feature_sets = {
    'solo_retornos': ['ret_1d', 'log_ret'],
    '8_features':    ['ret_1d', 'log_ret', 'vol_5d', 'vol_20d',
                      'vol_ratio', 'corr_30d', 'zscore', 'vol_ratio_v'],
}

ablation_results = {}

for name, cols in feature_sets.items():
    print(f'\n=== Entrenando: {name} ===')
    df_ab = build_features(close, volume, TICKER)[cols]
    X_tr, X_v, X_te, _, _, d_te, _ = split_and_scale(df_ab)

    m_ab, _ = train_model(
        X_tr, X_v,
        model_path=f'{BASE}/models/ablation_{name}.pt',
        epochs=50,
    )
    err = reconstruction_error(m_ab, X_te)
    thr = compute_threshold(reconstruction_error(m_ab, X_tr))
    ablation_results[name] = {'errors': err, 'dates': d_te, 'threshold': thr}
    print(f'[{name}] threshold={thr:.5f} | anomalias={(err > thr).sum()}')

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 8), sharex=True)
for ax, (name, data) in zip(axes, ablation_results.items()):
    ax.plot(data['dates'], data['errors'], lw=0.8, color='steelblue')
    ax.axhline(data['threshold'], color='crimson', ls='--', lw=1.2,
               label=f'Umbral 3σ = {data["threshold"]:.4f}')
    ax.fill_between(data['dates'],
                    data['errors'],
                    data['threshold'],
                    where=data['errors'] > data['threshold'],
                    color='crimson', alpha=0.3, label='Anomalía')
    ax.set_title(f'Ablation — {name}')
    ax.set_ylabel('MAE')
    ax.legend()
plt.tight_layout()
plt.savefig(f'{BASE}/results/ablation_study.png', dpi=150)
plt.show()

## 7. Resumen de resultados

In [ ]:
print('=' * 50)
print('RESUMEN DEL PIPELINE')
print('=' * 50)
print(f'Ticker analizado  : {TICKER}')
print(f'Datos descargados : {len(close)} dias ({close.index.min().date()} a {close.index.max().date()})')
print(f'Features          : {list(df.columns)}')
print(f'Split             : Train={X_train.shape} | Val={X_val.shape} | Test={X_test.shape}')
print(f'Mejor val_loss    : {min(history["val_loss"]):.6f}')
print()
print('Experimento 1 — Sigma sweep:')
print(sweep_df.to_string(index=False))
print()
print(f'Experimento 2 — Anomalias con sigma=3: {len(anomaly_dates)}')
print(f'Primera anomalia detectada: {anomaly_dates[0].strftime("%Y-%m-%d")} (PASO 2019)')
print()
print('Experimento 3 — Ablation study:')
for name, data in ablation_results.items():
    n = (data['errors'] > data['threshold']).sum()
    print(f'  {name:20s}: threshold={data["threshold"]:.5f} | anomalias={n}')
print()
print('Archivos guardados en:', BASE)